## Import Libraries & Setup

In [70]:
# Core Libraries
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Machine Learning & Deep Learning
import torch
import torch.nn as nn
from torchvision import transforms, models
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import faiss
import pickle
import joblib

# For image processing
import cv2
from pathlib import Path
import glob

# Visualization settings
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.9.1
CUDA available: False


In [71]:
# Visualization settings
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.9.1
CUDA available: False


## Define File Paths & Dataset Information

In [72]:
# Define paths
BASE_DIR = Path.cwd().parent.parent  # Navigate to car_sales/
DATA_DIR = BASE_DIR / 'development' / 'database'
MODELS_DIR = BASE_DIR / 'development' / 'models'
NOTEBOOK_DIR = BASE_DIR / 'development' / 'data_science'

def load_existing_models():
    """Load pre-trained sales and quantity prediction models using joblib"""
    
    models_loaded = {}
    
    # Quantity prediction models
    qty_dir = MODELS_DIR / 'quantity_prediction'
    if qty_dir.exists():
        print("Loading Quantity Prediction Models...")
        
        # Load scalers & encoders with joblib (handles scikit-learn objects robustly)
        models_loaded['qty_feature_scaler'] = joblib.load(qty_dir / 'scalers' / 'feature_scaler.pkl')
        models_loaded['qty_target_scaler'] = joblib.load(qty_dir / 'scalers' / 'target_scaler.pkl')
        models_loaded['qty_label_encoders'] = joblib.load(qty_dir / 'encoders' / 'label_encoders.pkl')
        
        # Load models
        models_loaded['qty_xgboost'] = joblib.load(qty_dir / 'models' / 'xgboost.pkl')
        models_loaded['qty_random_forest'] = joblib.load(qty_dir / 'models' / 'random_forest.pkl')
        models_loaded['qty_decision_tree'] = joblib.load(qty_dir / 'models' / 'decision_tree.pkl')
        
        # Load metadata
        with open(qty_dir / 'parameters' / 'feature_columns.json', 'r') as f:
            models_loaded['qty_feature_columns'] = json.load(f)
        
        metrics_df = pd.read_csv(qty_dir / 'metrics' / 'model_metrics.csv')
        models_loaded['qty_metrics'] = metrics_df
        
        print(f"  ✅ Quantity models loaded: {len(metrics_df)} models")
    
    # Sales prediction models
    sales_dir = MODELS_DIR / 'sales_prediction'
    if sales_dir.exists():
        print("\nLoading Sales Prediction Models...")
        
        models_loaded['sales_feature_scaler'] = joblib.load(sales_dir / 'scalers' / 'feature_scaler.pkl')
        models_loaded['sales_target_scaler'] = joblib.load(sales_dir / 'scalers' / 'target_scaler.pkl')
        models_loaded['sales_label_encoders'] = joblib.load(sales_dir / 'encoders' / 'label_encoders.pkl')
        
        models_loaded['sales_xgboost'] = joblib.load(sales_dir / 'models' / 'xgboost.pkl')
        models_loaded['sales_random_forest'] = joblib.load(sales_dir / 'models' / 'random_forest.pkl')
        models_loaded['sales_decision_tree'] = joblib.load(sales_dir / 'models' / 'decision_tree.pkl')
        
        with open(sales_dir / 'parameters' / 'feature_columns.json', 'r') as f:
            models_loaded['sales_feature_columns'] = json.load(f)
        
        metrics_df = pd.read_csv(sales_dir / 'metrics' / 'model_metrics.csv')
        models_loaded['sales_metrics'] = metrics_df
        
        print(f"  ✅ Sales models loaded: {len(metrics_df)} models")
        
    return models_loaded

# Load all existing models and scalers
existing_models = load_existing_models()
print("\nAll models and scalers loaded successfully.")
print(f"Total models loaded: {len(existing_models)}")

Loading Quantity Prediction Models...
  ✅ Quantity models loaded: 4 models

Loading Sales Prediction Models...
  ✅ Sales models loaded: 4 models

All models and scalers loaded successfully.
Total models loaded: 16


## Load database

In [73]:
# Load sales data
df_sales = pd.read_parquet(DATA_DIR / 'car_sales_prediction_sales.parquet')
df_quantity = pd.read_parquet(DATA_DIR / 'car_sales_prediction_quantity.parquet')
print(f"Sales DataFrame shape: {df_sales.shape}")
print(f"Quantity DataFrame shape: {df_quantity.shape}")

Sales DataFrame shape: (23905, 43)
Quantity DataFrame shape: (23905, 37)


In [74]:
# Load car images data
train_dir = DATA_DIR / 'Cars_Dataset' / 'train'
test_dir = DATA_DIR / 'Cars_Dataset' / 'test'

# Get image data
def get_image_data(directory):
    """Load images and corresponding labels from a directory."""
    image_data = []
    for brand in os.listdir(directory):
        brand_path = directory / brand
        if brand_path.is_dir():
            for img_path in brand_path.glob('*.jpg'):
                image_data.append({
                    'path': str(img_path),
                    'brand': brand,
                    'filename': img_path.name
                })
    return pd.DataFrame(image_data)

train_images_df = get_image_data(train_dir)
test_images_df = get_image_data(test_dir)

print("\nImage Dataset Summary:")
print("="*50)
print(f"Training images: {len(train_images_df)}")
print(f"Test images: {len(test_images_df)}")
print("\nTraining images by brand:")
print(train_images_df['brand'].value_counts())


Image Dataset Summary:
Training images: 3352
Test images: 813

Training images by brand:
brand
Audi                814
Toyota_Innova       775
Tata_Safari         441
Swift               424
Mahindra_Scorpio    316
Rolls_Royce         311
Hyundai_Creta       271
Name: count, dtype: int64


In [75]:
# Check brand overlap with sales and quantity datasets
image_brands = train_images_df['brand'].unique()
sales_brands = df_sales['company'].unique()
quantity_brands = df_quantity['company'].unique()
common_brands = set(image_brands) & set(sales_brands) & set(quantity_brands)

print("\nBrand Overlap Summary:")
print("="*50)
print(f"Image brands: {len(image_brands)}")
print(f"Sales brands: {len(sales_brands)}")
print(f"Quantity brands: {len(quantity_brands)}")
print(f"Common brands across all datasets: {len(common_brands)}")


Brand Overlap Summary:
Image brands: 7
Sales brands: 30
Quantity brands: 30
Common brands across all datasets: 1


## Vision transformer feature extractor

In [ ]:
from torchvision.models import ViT_B_16_Weights

class CarFeatureExtractor:
    def __init__(self, model_name='vit_base_patch16_224'):
        """Initialize the feature extractor with a pre-trained model."""
        self.device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

        # Load pre-trained ViT
        if model_name == 'vit_base_patch16_224':
            self.model = models.vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        else:
            self.model = models.vit_b_16(weights=ViT_B_16_Weights.DEFAULT)  # Default to ViT base if unknown model

        # use the classification head with identity to output the 768-dim class token instead
        self.model.heads = nn.Identity()
        
        self.model = self.model.to(self.device)
        self.model.eval()  # Set to evaluation mode

        # Image preprocessing transformations
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        print(f"✅ Vision Transformer initialized")
        print(f"   Device: {self.device}")
        print(f"   Model: {model_name}")

    def extract_features(self, image_path):
        """Extract features from image path"""
        try:
            image = Image.open(image_path).convert('RGB')
            image_tensor = self.transform(image).unsqueeze(0).to(self.device)

            with torch.no_grad():
                features = self.model(image_tensor)
                features = features.squeeze(0).cpu().numpy()  # Extract 1D array (768,)

            return features
        
        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            return None
        
    def extract_batch_features(self, image_paths, batch_size=32):
        """Extract features for multiple images in batches"""
        feature_list = []
        valid_paths = []

        for i, path in enumerate(image_paths):
            if (i + 1) % 50 == 0:
                print(f"Processed {i + 1}/{len(image_paths)} images")

            feat = self.extract_features(path)
            if feat is not None:
                feature_list.append(feat)
                valid_paths.append(path)

        return np.array(feature_list), valid_paths
    
# Initialize the feature extractor
extractor = CarFeatureExtractor()

✅ Vision Transformer initialized
   Device: mps
   Model: vit_base_patch16_224


In [77]:
# Test on sample image
sample_image = train_images_df['path'].iloc[0]
sample_features = extractor.extract_features(sample_image)
print(f"\nSample feature: {sample_features}")


Sample feature: [-3.28400731e-01  2.10973561e-01 -1.13016404e-01  1.79648772e-01
  5.10543525e-01 -6.10914052e-01 -2.40228161e-01  2.79249281e-01
 -2.38260016e-01 -3.73708069e-01  3.38702023e-01 -5.46540737e-01
 -1.38147163e+00  4.52859104e-01  1.34370271e-02 -1.04798448e+00
 -2.28282049e-01  9.10925046e-02 -6.16484165e-01 -3.58127952e-01
  7.36969292e-01  3.34974498e-01 -2.55467266e-01 -4.73314226e-02
  3.39446813e-01  7.37281978e-01  5.52738249e-01  2.72570290e-02
 -1.92519724e-01  3.86990905e-01 -2.82872200e-01 -6.51232481e-01
  1.99607939e-01  1.24777603e+00 -5.74318945e-01 -3.34441572e-01
 -1.69083685e-01 -6.23600304e-01 -5.01520336e-01 -2.85627842e-01
 -1.28425643e-01 -1.85239121e-01  7.46424019e-01 -4.95300740e-01
  1.00531650e+00  6.87747747e-02 -4.33282584e-01 -1.69582799e-01
 -8.42910051e-01  9.94602405e-03  3.22050929e-01  2.44261891e-01
 -3.83438826e-01  2.89479196e-01  2.98353404e-01  3.92561778e-02
 -2.88292557e-01 -6.33918345e-01 -1.40498030e+00 -6.69734657e-01
 -6.7383

## Extract image features

In [78]:
print("Extracting features from training images...")
print("="*50)

# sample subset for demonstration
max_images_per_brand = 100
sampled_images = []

for brand in train_images_df['brand'].unique():
    brand_images = train_images_df[train_images_df['brand'] == brand]
    if len(brand_images) > max_images_per_brand:
        brand_images = brand_images.sample(n=max_images_per_brand, random_state=42)
    sampled_images.extend(brand_images['path'].tolist())

print(f"Processing {len(sampled_images)} images for feature extraction...")

Extracting features from training images...
Processing 700 images for feature extraction...


In [79]:
# Extract features
image_features, valid_paths = extractor.extract_batch_features(sampled_images)

# Create feature dataframe
feature_df = pd.DataFrame({
    'path': valid_paths,
    'brand': [Path(p).parent.name for p in valid_paths]
})

# Add feature columns
for i in range(image_features.shape[1]):
    feature_df[f'feature_{i}'] = image_features[:, i]


print(f"\n✅ Feature extraction completed.")
print(f" Total images processed: {len(valid_paths)}")
print(f" Feature dimension: {image_features.shape[1]}")
print(f" Feature dataframe shape: {feature_df.shape}")

# Display
print("\nSample of extracted features:")
feature_df.head()

Processed 50/700 images
Processed 100/700 images
Processed 150/700 images
Processed 200/700 images
Processed 250/700 images
Processed 300/700 images
Processed 350/700 images
Processed 400/700 images
Processed 450/700 images
Processed 500/700 images
Processed 550/700 images
Processed 600/700 images
Processed 650/700 images
Processed 700/700 images

✅ Feature extraction completed.
 Total images processed: 700
 Feature dimension: 768
 Feature dataframe shape: (700, 770)

Sample of extracted features:


,path,brand,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_79,feature_80,feature_81,feature_82,feature_83,feature_84,feature_85,feature_86,feature_87,feature_88,feature_89,feature_90,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100,feature_101,feature_102,feature_103,feature_104,feature_105,feature_106,feature_107,feature_108,feature_109,feature_110,feature_111,feature_112,feature_113,feature_114,feature_115,feature_116,feature_117,feature_118,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128,feature_129,feature_130,feature_131,feature_132,feature_133,feature_134,feature_135,feature_136,feature_137,feature_138,feature_139,feature_140,feature_141,feature_142,feature_143,feature_144,feature_145,feature_146,feature_147,feature_148,feature_149,feature_150,feature_151,feature_152,feature_153,feature_154,feature_155,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,feature_166,feature_167,feature_168,feature_169,feature_170,feature_171,feature_172,feature_173,feature_174,feature_175,feature_176,feature_177,feature_178,feature_179,feature_180,feature_181,feature_182,feature_183,feature_184,feature_185,feature_186,feature_187,feature_188,feature_189,feature_190,feature_191,feature_192,feature_193,feature_194,feature_195,feature_196,feature_197,feature_198,feature_199,feature_200,feature_201,feature_202,feature_203,feature_204,feature_205,feature_206,feature_207,feature_208,feature_209,feature_210,feature_211,feature_212,feature_213,feature_214,feature_215,feature_216,feature_217,feature_218,feature_219,feature_220,feature_221,feature_222,feature_223,feature_224,feature_225,feature_226,feature_227,feature_228,feature_229,feature_230,feature_231,feature_232,feature_233,feature_234,feature_235,feature_236,feature_237,feature_238,feature_239,feature_240,feature_241,feature_242,feature_243,feature_244,feature_245,feature_246,feature_247,feature_248,feature_249,feature_250,feature_251,feature_252,feature_253,feature_254,feature_255,feature_256,feature_257,feature_258,feature_259,feature_260,feature_261,feature_262,feature_263,feature_264,feature_265,feature_266,feature_267,feature_268,feature_269,feature_270,feature_271,feature_272,feature_273,feature_274,feature_275,feature_276,feature_277,feature_278,feature_279,feature_280,feature_281,feature_282,feature_283,feature_284,feature_285,feature_286,feature_287,feature_288,feature_289,feature_290,feature_291,feature_292,feature_293,feature_294,feature_295,feature_296,feature_297,feature_298,feature_299,feature_300,feature_301,feature_302,feature_303,feature_304,feature_305,feature_306,feature_307,feature_308,feature_309,feature_310,feature_311,feature_312,feature_313,feature_314,feature_315,feature_316,feature_317,feature_318,feature_319,feature_320,feature_321,feature_322,feature_323,feature_324,feature_325,feature_326,feature_327,feature_328,feature_329,feature_330,feature_331,feature_332,feature_333,feature_334,feature_335,feature_336,feature_337,feature_338,feature_339,feature_340,featur

## Build FAISS index for Similarity Search

In [ ]:
class CarSimilaritySearch:
    def __init__(self, feature_df, feature_matrix):
        self.feature_df = feature_df
        self.feature_matrix = feature_matrix
        self.dimension = feature_matrix.shape[1]

        # Normalize features
        self._normalize_features()

        # Build FAISS index
        self._build_faiss_index()

        print(f"✅ FAISS index built!")
        print(f"   Dimension: {self.dimension}")
        print(f"   Total vectors: {self.index.ntotal}")

    def _normalize_features(self):
        """Normalize features to unit vectors for cosine similarity"""
        norms = np.linalg.norm(self.feature_matrix, axis=1, keepdims=True)
        self.feature_matrix_norm = self.feature_matrix / (norms + 1e-8) # Avoid division by zero

    def _build_faiss_index(self):
        """Build FAISS index for fast similarity search"""
        self.index = faiss.IndexFlatIP(self.dimension)  # Inner product for cosine similarity
        self.index.add(self.feature_matrix_norm.astype(np.float32))

    def search(self, query_features, k=5):
        """Search for similar images given query features"""
        # Normalize query
        query_norm = query_features / (np.linalg.norm(query_features) + 1e-8)
        query_norm = query_norm.reshape(1, -1).astype(np.float32)
        
        # Search
        distances, indices = self.index.search(query_norm, k)

        # Get results
        results = []
        for idx, score in zip(indices[0], distances[0]):
            if idx < len(self.feature_df):
                results.append({
                    'path': self.feature_df.iloc[idx]['path'],
                    'brand': self.feature_df.iloc[idx]['brand'],
                    'similarity_score': float(score),
                    'index': idx
                })
        
        return results